# МТС · Проверка Qwen3.5-0.8B до обучения

Выполните ячейки **1–6 по очереди**, затем скачайте результат ячейкой 8. Модель обрабатывает первые 10 официальных val-звонков. Обучения в блокноте нет.

Загрузите новый архив **mts_qwen35_probe.zip**. Результаты сохраняются отдельно от Qwen3-0.6B.

Изменены модель, инструкция и обработка Markdown. Это проверка новой конфигурации, а не изолированное доказательство, что одна архитектура лучше другой. Для честного сравнения моделей нужно использовать одинаковые prompt и правила.


## 1. Проверить GPU
В меню выберите: **Среда выполнения → Сменить среду выполнения → GPU**. Доступ к GPU не гарантируется Colab.

In [ ]:
from pathlib import Path
import json, sys, subprocess
import torch

ROOT = Path("/content/mts_qwen35_probe")
print("GPU доступна:", torch.cuda.is_available())
assert torch.cuda.is_available(), "Выберите GPU в настройках среды выполнения. Дальше пока не запускайте."
print("Видеокарта:", torch.cuda.get_device_name(0))
print("VRAM, ГБ:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
print("Torch:", torch.__version__)

## 2. Загрузить ZIP с проектом
Нажмите кнопку выбора файла и укажите **mts_qwen35_probe.zip**. Это не архив старого учебного чат-бота.

In [ ]:
from google.colab import files
import zipfile, hashlib

uploaded = files.upload()
archive = next((name for name in uploaded if name == "mts_qwen35_probe.zip"), None)
if archive is None:
    matches = [name for name in uploaded if name.lower().endswith(".zip")]
    assert len(matches) == 1, "Выберите один архив mts_qwen35_probe.zip"
    archive = matches[0]
extract_base = Path("/content").resolve()
with zipfile.ZipFile(archive) as z:
    for info in z.infolist():
        target = (extract_base / info.filename).resolve()
        assert target.is_relative_to(extract_base), "Недопустимый путь в архиве"
    z.extractall(extract_base)
assert (ROOT / "predict.py").is_file(), "Структура архива не соответствует проекту"
expected = json.loads((ROOT / "official_hashes.json").read_text())
for name, digest in expected.items():
    actual = hashlib.sha256((ROOT / "data" / name).read_bytes()).hexdigest()
    assert actual == digest, f"Файл повреждён при загрузке: {name}"
del uploaded
print("Проект распакован:", ROOT)
print("Все 8 официальных файлов совпадают по хешам.")

## 3. Установить зависимости и проверить данные
Сейчас обучение и модель ещё не запускаются. Предупреждение о расхождении baseline — уже выявленная особенность пакета МТС.

In [ ]:
print("Установка Transformers и Accelerate...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(ROOT / "requirements.txt")], check=True)
print("Зависимости установлены. Проверяем поддержку модели...")
subprocess.run([sys.executable, "-c", "from transformers import AutoModelForImageTextToText; from transformers.models.qwen3_5 import Qwen3_5ForConditionalGeneration; print('Qwen3.5 поддерживается установленной библиотекой')"], check=True)
print("Проверяем данные...")
subprocess.run([sys.executable, str(ROOT / "audit_data.py"), "--data", str(ROOT / "data"), "--out", str(ROOT / "audit.json")], check=True)
print("Аудит завершён. Следующий шаг — сохранение результатов на Drive.")

## 4. Подключить постоянное хранилище
При подключении Drive подтвердите доступ в интерфейсе Google. Ответы сохраняются после каждого звонка. Смена типа GPU или настроек требует нового `RUN_TAG`, чтобы не смешивать результаты.

In [ ]:
USE_GOOGLE_DRIVE = True
RUN_TAG = "qwen35_0_8b_before_v1"
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    RESULT_BASE = Path("/content/drive/MyDrive/MTS_CompactFactExtraction/qwen35_probe")
else:
    RESULT_BASE = ROOT / "results"
    print("ВНИМАНИЕ: временный диск Colab. Результаты могут исчезнуть после отключения.")
RUN = RESULT_BASE / RUN_TAG
RUN.mkdir(parents=True, exist_ok=True)
import shutil
shutil.copy2(ROOT / "audit.json", RUN / "audit.json")
print("Результаты будут здесь:", RUN)

## 5. Получить ответы исходной Qwen на 10 звонках
При первом запуске скачаются веса модели. Дождитесь конца ячейки; не запускайте её параллельно. Слабые ответы исходной модели — результат эксперимента, а не ошибка установки. Невалидный JSON и слишком длинные входы сохраняются со статусом ошибки.

Предупреждения о PyTorch fallback для causal_conv1d/flash-linear-attention допустимы: это медленный путь выполнения, а не ошибка извлечения. Первый тест не устанавливает эти ускорители.

In [ ]:
subprocess.run([
    sys.executable, "-u", str(ROOT / "predict.py"),
    "--data", str(ROOT / "data"), "--out", str(RUN), "--limit", "10"
], check=True)

def evaluate_current_run():
    for prediction, output in [
        ("pred_before.jsonl", "metrics_before.json"),
        ("baseline_selected.jsonl", "metrics_mts_selected.json"),
    ]:
        subprocess.run([
            sys.executable, str(ROOT / "data" / "case_metrics.py"),
            str(RUN / "gold_evaluation.jsonl"), str(RUN / prediction),
            "--json", str(RUN / output)
        ], check=True)
evaluate_current_run()
print("Готово: проверены первые 10 звонков. Теперь выполните ячейку 6.")

## 6. Посмотреть результат — остановиться здесь
Сохраните `summary.json` и `metrics_before.json` или покажите их содержимое. Сначала разберём работоспособность и ошибки; показатели 10 звонков не представляют весь корпус.

In [ ]:
for name in ["summary.json", "metrics_before.json", "metrics_mts_selected.json"]:
    print("\n" + name)
    print((RUN / name).read_text(encoding="utf-8"))
manifest = json.loads((RUN / "run_manifest.json").read_text())
print("\nФактическое полное число параметров:", manifest.get("total_parameters"))
preview = json.loads((RUN / "preview.json").read_text(encoding="utf-8"))
if preview:
    ex = preview[0]
    print("\nПример звонка:", ex["call_id"], "Статус:", ex["status"])
    for turn in ex["turns"]:
        print(f"#{turn['idx']} {turn['speaker']}: {turn['text']}")
    print("\nПРАВИЛЬНЫЕ ФАКТЫ (для просмотра, не вход модели):")
    print(json.dumps(ex["expected_facts"], ensure_ascii=False, indent=2))
    print("\nФАКТЫ QWEN:")
    print(json.dumps(ex["predicted_facts"], ensure_ascii=False, indent=2))
    if ex["error"]:
        print("Ошибка:", ex["error"])
        print("Сырой ответ:", ex["raw"][:6000])

## 7. Позже: полный прогон 383 звонков
По умолчанию отключён. Когда первые ответы проверены, замените `False` на `True` и запустите эту ячейку. Она использует сохранённые первые 10 записей при совпадении настроек; остальные обрабатывает постепенно.

In [ ]:
RUN_FULL_VALIDATION = False
if RUN_FULL_VALIDATION:
    subprocess.run([
        sys.executable, "-u", str(ROOT / "predict.py"),
        "--data", str(ROOT / "data"), "--out", str(RUN)
    ], check=True)
    evaluate_current_run()
    print((RUN / "summary.json").read_text(encoding="utf-8"))
else:
    print("Полный прогон отключён. Сначала проверяем результат 10 звонков.")

## 8. Скачать результаты
Можно выполнить сразу после ячейки 6. Архив содержит ответы, метрики, ошибки и настройки. Дообученных весов на этом этапе ещё нет.

In [ ]:
from google.colab import files
export_path = Path("/content/mts_qwen35_probe_results.zip")
with zipfile.ZipFile(export_path, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for path in RUN.rglob("*"):
        if path.is_file() and not path.name.endswith(".tmp"):
            z.write(path, path.relative_to(RUN))
print("Скачайте архив результатов:", export_path)
files.download(str(export_path))

## Следующий этап

Когда получены и проверены исходные ответы, подготовим LoRA-обучение на официальном train и повторим оценку на тех же звонках. Затем отдельно добавим проверенные правила дат/состояний, Docker API, CPU/GPU-замеры и три режима по ТЗ.

**Не обучать на val. Не считать ошибки/пустые fallback успешными ответами. Не сравнивать качество на 10 звонках с контрольными метриками на 383.** В этом блокноте `baseline_selected.jsonl` формируется для той же выборки.

Подробности: README_START.md в архиве. GPU-прогон этого блокнота ещё нужно выполнить; локально проверены исходные данные и код без загрузки весов.
